## 1.0 Connecting to Google Drive

This notebook is designed to provide an accesible platform for analysing sequencing data. This notebook allows you to perfom analysis without the need to dowload or install any additional software on your computer. You only need is a Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rpy2
%load_ext rpy2.ipython
!git clone https://github.com/stijnteunissen/Workshop_H2Omics_test.git

# 2.0 Analysis Preparation

### 2.1 Creating the Output Folder

This code block creates a directory in Google Drive within the home folder (`"drive/MyDrive/"`). The new directory will be created after choosing a self-defined project name, and then uses that to name for the directory. This directory will be used to store all output files generated during the workshop, ensuring they are organized and saved in your Google Drive. After running the code, write a project name (use underscores "_" instead of spaces), and press confirm for the project name.

Opening Google drive (https://drive.google.com/drive/my-drive)

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from starting_project import starting_project

starting_project()

### 2.2 Loading Input Files

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from norm_and_import import norm_and_import

norm_and_import()

### 2.3 Installing and Loading R Packages

In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_workshop/install_packages.R")

install_packages()

# 3.0 Starting the analysis

### 3.1 copy number prediction

In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_workshop/copy_number_prediction.R")

copy_number_prediction()

### 3.2 Combining all data into a phyloseq object


In [ ]:
%%R
if(!dir.exists(paste0(base_path, projects, "/messages"))){dir.create(paste0(base_path, projects, "/messages"))}
log_file = paste0(base_path, glue("{projects}/messages/logging_output.txt"))

# create folders and copy files
H2Omics::create_folders(projects)

# combine qiime2 metadata with experimental sampled metadata
unified_metadata = H2Omics::unify_metadata(projects)

# create a physeq object
physeq = H2Omics::creating_physeq_object(projects)

### 3.3 Optimising the taxonomic information

In [ ]:
%%R
# tax clean
options(width = 140)
cleaned_physeq = H2Omics::tax_clean(physeq = physeq, tax_filter = TRUE)

### 3.4 Removing contaminant and Mock community ASVs

In [ ]:
%%R
# Resolve polytomous branching of the QIIME2 Fasttree2 phylogeny into a fully bifurcated tree for phylogenetic analyses
resolved_tree_physeq = H2Omics::resolve_tree(physeq = cleaned_physeq)

# decontam (decon_method = frequency, prevalence or both)
decontam_physeq = H2Omics::decontam(physeq = resolved_tree_physeq, decon_method = both, blank = TRUE)

# remove mock and mock features
without_mock_physeq = H2Omics::remove_mock(physeq = decontam_physeq, mock_genera = mock_genera, mock = TRUE)

### 3.5 Normalisation of microbial data
*   Ribosomal gene copy number correction
*   Biomas normalisation of microbiome
*   Rarefaction

In [ ]:
%%R
# copy number correction and biomasss normalisation
normalised_asv_physeq = H2Omics::normalise_data(physeq = without_mock_physeq, norm_method = norm_method, copy_correction = TRUE)

# rarefied data
rarefied_asv_physeq = H2Omics::rarefying(physeq = normalised_asv_physeq, norm_method = norm_method, iteration = 10)

rarefied_tax_physeq = H2Omics::group_tax(physeq = rarefied_asv_physeq, norm_method = norm_method)

# converting phyloseq object to a tibble
rarefied_tax_psmelt = H2Omics::psdata_to_tibble(physeq = rarefied_tax_physeq, norm_method = norm_method)

### 3.6 Creating Relative barplot

In [ ]:
%%R -w 10 -h 8 -u in
# Relative Barplot
created_barplot = H2Omics::barplot(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

### 3.7 Creating Absolute barplot

In [ ]:
%%R -w 10 -h 8 -u in
# Absolute Barplot
created_barplot = H2Omics::barplot2(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

### 3.8 Creating a Relative barplot for pathogenic bacteria

In [ ]:
%%R -w 10 -h 8 -u in
# pathogens genus
H2Omics::barplot_extra(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

### 3.9 Creating a Absolute barplot for pathogenic bacteria

In [ ]:
%%R -w 10 -h 8 -u in
# pathogens genus
options(width = 140)
H2Omics::barplot_extra2(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

### 3.10 Creating a Heatmap

In [ ]:
%%R -w 10 -h 8 -u in
# heatmap
heatmap_plot = H2Omics::heatmap(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method)

### 3.11 Alpha diversity: the bacterial richness and diversity within samples

In [ ]:
%%R -w 12 -h 7 -u in
# alpha diversity
options(width = 120)
alpha_div_plots = H2Omics::alpha_diversity(physeq = rarefied_asv_physeq, taxrank = "Tax_label", norm_method = norm_method)

### 3.12 Beta diversity: pairwise comparing microbiome samples


In [ ]:
%%R -w 8 -h 6 -u in
# beta diversity
beta_div_plots = H2Omics::beta_diversity(physeq = rarefied_asv_physeq, taxrank = "Tax_label", norm_method = norm_method,
                                         ordination_method = "PCoA", color_factor = "treatment", color_continuous = FALSE,
                                         shape_factor = "timepoint", size_factor = NULL, alpha_factor = NULL)

### 3.13 Exporting results and figures

In [ ]:
%%R
# export data
H2Omics::export_data()

After exporting the data, the workshop is complete. However, if you would like to perform the analysis with your own data, you can follow this link (). It leads to a notebook specifically designed for analyzing your own dataset. The notebook explains each step in detail, including the required input files needed to conduct the analysis. It is based on 16S Illumina data.

If you would like to use the notebook with Nanopore data instead, you can try this link (https://github.com/timyerg/NaMeco) to convert your Nanopore data into .qza files so it can be used with this notebook. Please note that we have not tested this process.